# Nüks (cured vs recurrence) modeli — 5-fold çapraz doğrulama

Bu not defteri **Google Colab** üzerinde çalışır. Bilgisayarına hiçbir şey kurman gerekmez.

Hücreleri **sırayla** çalıştır: her hücrenin solundaki ▶ düğmesine bas, bitmesini bekle, sonra bir sonrakine geç.

---
### Başlamadan önce: GPU'yu aç
Üst menüden **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**.
GPU olmadan da çalışır, sadece daha yavaş olur.


## 1. GPU bağlı mı kontrol et


In [ ]:
import torch
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU YOK - CPU ile calisacak (daha yavas).')
    print('Runtime > Change runtime type > T4 GPU secip tekrar dene.')


## 2. Google Drive'ı bağla

Çalıştırınca bir link çıkar: Google hesabını seç, izin ver, çıkan kodu kutuya yapıştır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Veri klasörünün yolunu bul

Aşağıdaki hücre Drive'ında `train-test-folder` klasörünü arar.


In [ ]:
import os

hits = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'train-test-folder' in dirs:
        hits.append(os.path.join(root, 'train-test-folder'))
    if len(root.split(os.sep)) > 8:   # cok derine inme
        dirs[:] = []

if hits:
    print('Bulunan klasor(ler):')
    for h in hits: print('  ', h)
    DATA = hits[0]
    print('\nKullanilacak:', DATA)
else:
    print('Bulunamadi. Sol taraftaki klasor ikonundan yolu bulup asagiya elle yaz.')
    DATA = '/content/drive/MyDrive/BURAYA/train-test-folder'


Klasör yapısı şöyle olmalı:
```
train-test-folder/
├── LVOT/{train_cured, train_notcured, test_cured, test_notcured}
├── RVOT/{...}
└── RLVOT/{...}
```


In [ ]:
for site in ['LVOT','RVOT','RLVOT']:
    p = os.path.join(DATA, site)
    if os.path.isdir(p):
        print(site, '->', sorted(os.listdir(p)))
    else:
        print(site, '-> YOK')


## 4. Kodu indir


In [ ]:
# Kendi repona push ettikten sonra asagidaki satiri kendi adresinle degistir:
REPO_URL = 'https://github.com/KULLANICI_ADIN/MultiQRSNet-Recurrence.git'

import os
if not os.path.exists('MultiQRSNet-Recurrence'):
    !git clone $REPO_URL
%cd MultiQRSNet-Recurrence
!ls


Repoyu henüz oluşturmadıysan, `recurrence_cv.py` dosyasını sol taraftaki klasör ikonuna sürükleyip bırak ve bu hücreyi atla.


## 5. Gerekli paketler
Colab'da hepsi zaten kurulu; bu hücre sadece sürümleri doğrular.


In [ ]:
!pip install -q 'scikit-learn>=0.24' scipy pandas numpy
from sklearn.model_selection import StratifiedGroupKFold
print('Hazir.')


## 6. ÖNCE BUNU ÇALIŞTIR — hasta gruplaması doğru mu?

**Bu adım atlanamaz.** Kodun leakage koruması, hangi kayıtların aynı hastaya ait olduğunu
doğru çözmesine bağlı.

Çıktıda şuna bak:
- `recordings/patient` **1'den büyük** olmalı (hasta başına birden fazla kayıt varsa).
- Eğer **her kayıt ayrı hasta** olduysa gruplama çalışmıyor demektir → aşağıdaki
  `--patient-id` seçeneklerini dene.
- `example paths -> patient id` satırları mantıklı görünmeli.


In [ ]:
!python recurrence_cv.py --data "$DATA" --sanity-check


### Hasta ID'si yanlış çözüldüyse
Aşağıdakilerden birini dene (satır başındaki `#` işaretini kaldır):


In [ ]:
# Hasta = dosyanin bulundugu klasor adi
# !python recurrence_cv.py --data "$DATA" --sanity-check --patient-id parent

# Hasta = dosya adinin ilk _ veya - isaretine kadarki kismi
# !python recurrence_cv.py --data "$DATA" --sanity-check --patient-id stem

# Kendi kalibin: ornegin dosya adi 'HASTA042_kayit3.txt' ise
# !python recurrence_cv.py --data "$DATA" --sanity-check --patient-id regex --patient-pattern '(HASTA\d+)'


## 7. Eğitim

Yukarıdaki çıktı doğru göründüyse burayı çalıştır. GPU ile birkaç dakika sürer.


In [ ]:
!python recurrence_cv.py --data "$DATA" --out results_recurrence --epochs 60 --seed 42


## 8. Sonuç tabloları

Bunlar doğrudan Supplementary Tablo S2 ve S3'e girer.


In [ ]:
import pandas as pd, json

print('=== Tablo S2: fold kompozisyonu ===')
display(pd.read_csv('results_recurrence/fold_composition.csv'))

print('=== Tablo S3: fold bazinda metrikler ===')
display(pd.read_csv('results_recurrence/fold_metrics.csv').round(4))

s = json.load(open('results_recurrence/summary.json'))
print('=== Havuzlanmis (fold ortalamasi, %95 GA) ===')
for k, v in s['per_fold_mean_95ci'].items():
    print(f"  {k:<10} {v['mean']:.4f}  (95% GA {v['lo']:.4f} - {v['hi']:.4f})")

print('\n=== HASTA DUZEYI (yazida birincil olarak bunu raporla) ===')
for k, v in s['pooled_oof_patient_level'].items():
    print(f'  {k:<10} {v}')
ci = s['patient_level_bootstrap_ci']
print(f"  AUC    %95 GA: {ci['auc'][0]:.4f} - {ci['auc'][1]:.4f}")
print(f"  PR-AUC %95 GA: {ci['pr_auc'][0]:.4f} - {ci['pr_auc'][1]:.4f}")

prev = s['n_recurrence_patients'] / s['n_patients']
print(f"\nNuks prevalansi (PR-AUC icin 'beceriksiz model' temeli): {prev:.4f}")
print('PR-AUC bu degerin belirgin uzerinde degilse model klinik bilgi katmiyor demektir.')


## 9. MDI ablasyonu (opsiyonel ama hakem için faydalı)

Hakem "SHAP'a göre MDI katkısı zayıf, neden tutuyorsunuz?" diye sordu.
MDI'sız çalıştırıp performansın değişip değişmediğine bak — cevabın bu.


In [ ]:
!python recurrence_cv.py --data "$DATA" --out results_no_mdi --epochs 60 --seed 42 --no-mdi

import json
a = json.load(open('results_recurrence/summary.json'))['pooled_oof_patient_level']
b = json.load(open('results_no_mdi/summary.json'))['pooled_oof_patient_level']
print(f"{'metrik':<10}{'MDI ile':>10}{'MDI siz':>10}")
for k in ['accuracy','f1','auc','pr_auc']:
    print(f'{k:<10}{a[k]:>10.4f}{b[k]:>10.4f}')


## 10. Farklı tohumlarla tekrar (küçük veride önemli)

Bu boyutta bir veri setinde tek bir çalıştırma yanıltıcı olabilir.
Birkaç tohumla çalıştırıp yayılımı raporlamak daha dürüst.


In [ ]:
import json
rows = []
for seed in [1, 2, 3, 42, 2024]:
    !python recurrence_cv.py --data "$DATA" --out results_seed$seed --epochs 60 --seed $seed > /dev/null 2>&1
    m = json.load(open(f'results_seed{seed}/summary.json'))['pooled_oof_patient_level']
    rows.append({'seed': seed, **{k: round(m[k], 4) for k in ['accuracy','f1','auc','pr_auc']}})

import pandas as pd
df = pd.DataFrame(rows)
display(df)
print('\nTohumlar arasi ortalama +/- SS:')
for k in ['accuracy','f1','auc','pr_auc']:
    print(f'  {k:<10}{df[k].mean():.4f} +/- {df[k].std():.4f}')


## 11. Sonuçları indir


In [ ]:
!zip -qr results.zip results_recurrence results_no_mdi
from google.colab import files
files.download('results.zip')
